In [1]:
#loading data & cleanding data

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import StandardScaler

data_directory = Path.cwd() / "csv_files"

dalys = pd.read_csv(data_directory / "dalys.CSV")
deaths = pd.read_csv(data_directory / "deaths.CSV")
ylds = pd.read_csv(data_directory / "ylds.CSV")
ylls = pd.read_csv(data_directory / "ylls.CSV")


#unique pathogen-drug combinations:
def base_filter(df):
    # Keep BOTH counterfactuals — split happens in Cell 2
    return df[
        (df["metric_name"] == "Count") &
        (df["sex_name"]    == "All") &
        (df["age_group_name"] == "All Ages") &
        (df["infectious_syndrome"] == "All infectious syndromes")
    ].copy()

deaths = base_filter(deaths)
dalys  = base_filter(dalys)
ylds   = base_filter(ylds)
ylls   = base_filter(ylls)

cols = ['location_name', 'pathogen', 'antibiotic_class', 'counterfactual', 'val']
deaths = deaths[cols]
dalys  = dalys[cols]
ylds   = ylds[cols]
ylls   = ylls[cols]

print("Deaths shape:", deaths.shape)
print("Counterfactuals present:", deaths['counterfactual'].unique())

# filtered_deaths kept for the exploratory plots in later cells
filtered_deaths = deaths[deaths['counterfactual'] == 'Drug-susceptible infection'].copy()

Deaths shape: (9144, 5)
Counterfactuals present: ['Drug-susceptible infection' 'No infection']


In [2]:
grp = ['location_name', 'pathogen', 'antibiotic_class']

deaths_resistant = (
    deaths[deaths['counterfactual'] == 'Drug-susceptible infection']
    .groupby(grp)['val'].sum()
    .rename('resistant_deaths')
)

deaths_total = (
    deaths[deaths['counterfactual'] == 'No infection']
    .groupby(grp)['val'].sum()
    .rename('total_deaths')
)

deaths_split = pd.concat([deaths_resistant, deaths_total], axis=1).reset_index()


dalys_resistant = (
    dalys[dalys['counterfactual'] == 'Drug-susceptible infection']
    .groupby(grp)['val'].sum()
    .rename('resistant_dalys')
)

dalys_total = (
    dalys[dalys['counterfactual'] == 'No infection']
    .groupby(grp)['val'].sum()
    .rename('total_dalys')
)

dalys_split = pd.concat([dalys_resistant, dalys_total], axis=1).reset_index()


ylds_resistant = (
    ylds[ylds['counterfactual'] == 'Drug-susceptible infection']
    .groupby(grp)['val'].sum()
    .rename('resistant_ylds')
)

ylds_total = (
    ylds[ylds['counterfactual'] == 'No infection']
    .groupby(grp)['val'].sum()
    .rename('total_ylds')
)

ylds_split = pd.concat([ylds_resistant, ylds_total], axis=1).reset_index()

ylls_resistant = (
    ylls[ylls['counterfactual'] == 'Drug-susceptible infection']
    .groupby(grp)['val'].sum()
    .rename('resistant_ylls')
)

ylls_total = (
    ylls[ylls['counterfactual'] == 'No infection']
    .groupby(grp)['val'].sum()
    .rename('total_ylls')
)

ylls_split = pd.concat([ylls_resistant, ylls_total], axis=1).reset_index()
print(deaths_split.head())

         location_name                 pathogen  \
0  Antigua and Barbuda  Acinetobacter baumannii   
1  Antigua and Barbuda  Acinetobacter baumannii   
2  Antigua and Barbuda  Acinetobacter baumannii   
3  Antigua and Barbuda  Acinetobacter baumannii   
4  Antigua and Barbuda  Acinetobacter baumannii   

                                    antibiotic_class  resistant_deaths  \
0                                    Aminoglycosides          0.249725   
1  Anti-pseudomonal penicillin/Beta-Lactamase inh...          0.139687   
2              Beta Lactam/Beta-lactamase inhibitors          0.006248   
3                                        Carbapenems          1.192866   
4                                   Fluoroquinolones          1.114843   

   total_deaths  
0      5.981997  
1      8.109166  
2      7.070885  
3      7.353833  
4      8.595456  


In [3]:
deaths_agg = deaths.groupby(['location_name','pathogen','antibiotic_class'])['val'] \
                   .sum().reset_index().rename(columns={'val':'total_deaths'})

feature_matrix = deaths_split.merge(dalys_split, on=grp)
feature_matrix = feature_matrix.merge(ylds_split, on=grp)
feature_matrix = feature_matrix.merge(ylls_split, on=grp)

# Remove "All pathogens" and "one or more" aggregate rows
feature_matrix = feature_matrix[
    ~feature_matrix['pathogen'].str.contains('All', case=False) &
    ~feature_matrix['antibiotic_class'].str.contains('one or more', case=False)
].copy()

# Clip negatives
resist_cols = ['resistant_deaths','resistant_dalys','resistant_ylds','resistant_ylls']
total_cols  = ['total_deaths','total_dalys','total_ylds','total_ylls']
feature_matrix[resist_cols + total_cols] = feature_matrix[resist_cols + total_cols].clip(lower=0)

print(f"Shape: {feature_matrix.shape}")
print(f"Columns: {list(feature_matrix.columns)}")

Shape: (3096, 11)
Columns: ['location_name', 'pathogen', 'antibiotic_class', 'resistant_deaths', 'total_deaths', 'resistant_dalys', 'total_dalys', 'resistant_ylds', 'total_ylds', 'resistant_ylls', 'total_ylls']


In [4]:
cols_4 = ['resistant_deaths', 'resistant_dalys', 'resistant_ylds', 'resistant_ylls']
print(feature_matrix[cols_4].describe())

correlation_4 = feature_matrix[cols_4].corr()
print("\nCorrelation (4 resistance-attributable features):")
print(correlation_4.round(4))

#motivation for pca

       resistant_deaths  resistant_dalys  resistant_ylds  resistant_ylls
count      3.096000e+03      3096.000000     3096.000000     3096.000000
mean       9.079843e+01      2248.499115       15.816020     2232.930070
std        6.080558e+02     14189.789872       97.256045    14107.040380
min        3.633304e-07         0.000016        0.000000        0.000016
25%        2.150306e-01         5.853045        0.032309        5.757637
50%        2.120514e+00        62.057750        0.349990       61.319623
75%        1.679345e+01       480.780691        3.078494      475.526554
max        2.341518e+04    531359.401243     2492.641658   528866.759585

Correlation (4 resistance-attributable features):
                  resistant_deaths  resistant_dalys  resistant_ylds  \
resistant_deaths            1.0000           0.9949          0.8521   
resistant_dalys             0.9949           1.0000          0.8540   
resistant_ylds              0.8521           0.8540          1.0000   
resistan

In [5]:
# reducing correlation, need independent features

# reduce dominant clustering by extreme countries
feature_matrix['log_burden'] = np.log10(feature_matrix['total_dalys'] + 1)

feature_matrix['mortality_prop'] = (feature_matrix['total_ylls'] /(feature_matrix['total_ylls'] + feature_matrix['total_ylds']))

feature_matrix['resistance_attr_frac'] = (feature_matrix['resistant_dalys'] / feature_matrix['total_dalys'] + 1e-9)

In [6]:
cols_3 = ['log_burden', 'mortality_prop', 'resistance_attr_frac']


print("3 independent features:")
print(feature_matrix[cols_3].describe().round(4))

print("Correlation:")
correlation_3 = feature_matrix[cols_3].corr()
print(correlation_3.round(4))

3 independent features:
       log_burden  mortality_prop  resistance_attr_frac
count   3096.0000       3096.0000             3096.0000
mean       2.7977          0.9765                0.1339
std        1.3623          0.0686                0.1045
min        0.0001          0.2986                0.0000
25%        1.8656          0.9858                0.0571
50%        2.8333          0.9923                0.1094
75%        3.7923          0.9951                0.1910
max        6.3547          0.9999                0.5962
Correlation:
                      log_burden  mortality_prop  resistance_attr_frac
log_burden                1.0000          0.2579               -0.2690
mortality_prop            0.2579          1.0000               -0.1289
resistance_attr_frac     -0.2690         -0.1289                1.0000


In [7]:
# Aggregate to (pathogen, antibiotic_class) for clustering
feature_matrix_agg = (
    feature_matrix
    .groupby(['pathogen', 'antibiotic_class'])
    .agg(
        resistant_dalys      = ('resistant_dalys', 'sum'),
        resistant_ylls       = ('resistant_ylls',  'sum'),
        resistant_ylds       = ('resistant_ylds',  'sum'),
        resistant_deaths     = ('resistant_deaths','sum'),
        total_dalys          = ('total_dalys',     'sum'),
    )
    .reset_index()
)

feature_matrix_agg['log_burden']          = np.log10(feature_matrix_agg['resistant_dalys'] + 1)
feature_matrix_agg['mortality_prop']      = (feature_matrix_agg['resistant_ylls'] /
                                              (feature_matrix_agg['resistant_ylls'] + feature_matrix_agg['resistant_ylds']))
feature_matrix_agg['resistance_attr_frac'] = (feature_matrix_agg['resistant_dalys'] /
                                               (feature_matrix_agg['total_dalys'] + 1e-9))

key_features = ['log_burden', 'mortality_prop', 'resistance_attr_frac']
X_log = feature_matrix_agg[key_features].copy()

sc = StandardScaler()
X_scaled = sc.fit_transform(X_log)

feature_matrix_agg.to_csv('amr_feature_matrix.csv', index=False)
np.save('amr_X_scaled.npy', X_scaled)

print(f"Saved amr_feature_matrix.csv  shape={feature_matrix_agg.shape}")
print(f"Saved amr_X_scaled.npy         shape={X_scaled.shape}")
print(f"Features used: {key_features}")

Saved amr_feature_matrix.csv  shape=(86, 10)
Saved amr_X_scaled.npy         shape=(86, 3)
Features used: ['log_burden', 'mortality_prop', 'resistance_attr_frac']
